In [ ]:
import os
os.environ["VLLM_CONFIGURE_LOGGING"] = "0"
import logging
logging.basicConfig(format='%(message)s', level=logging.FATAL+1)

import sys
sys.path.append("..")

import gc
import time

import torch
from vllm import LLM

from sal.config import Config

from core import bon_search_v1

from utils.load_data import load_data_hf

In [ ]:
base_dir = '/groups/chichengz/tnn/datasets/'

llm_dir = base_dir + "Llama3.2-3B-Instruct"

# dataset
ds_split = "test"
ds_dir = base_dir + "/prm800k/math_splits"

In [ ]:
# quantization configs to benchmark
# AWQ/GPTQ require a pre-quantized model directory
quant_configs = [
    {
        "name":         "fp16",
        "quantization": None,
        "load_format":  "auto",
        "dtype":        "float16",
    },
    {
        "name":         "int8 (bitsandbytes)",
        "quantization": "bitsandbytes",
        "load_format":  "bitsandbytes",
        "dtype":        "float16",
    },
    # {
    #     "name":         "awq",
    #     "quantization": "awq",
    #     "load_format":  "auto",
    #     "dtype":        "float16",
    # },
    # {
    #     "name":         "gptq",
    #     "quantization": "gptq",
    #     "load_format":  "auto",
    #     "dtype":        "float16",
    # },
]

In [ ]:
# general params
config = Config()
config.agg_strategy = 'last'
config.temperature = 0.8
config.max_tokens = 2048

config.n = 32
config.filter_duplicates = True
config.date_string = "Aug 1 2025"
config.seed = 0

num_trials = 2
level = 4

llm_gpu_memory_utilization = 0.9

In [ ]:
dataset = load_data_hf(ds_dir, ds_split=ds_split, level=level)

num_questions = len(dataset)
batch_of_questions = [dataset[i]['question'] for i in range(num_questions)]
print(f"num_questions = {num_questions}")

### Benchmark `best_of_n_v1` across quantization levels
Each config is loaded, timed, then unloaded before the next to avoid OOM.

In [ ]:
def get_gpu_memory_used(device=0):
    free, total = torch.cuda.mem_get_info(device)
    return (total - free) / (1024**3)


results_summary = []

for qcfg in quant_configs:
    print(f"\n=== {qcfg['name']} ===")

    llm_vllm = LLM(
        model=llm_dir,
        tensor_parallel_size=1,
        max_model_len=5000,
        gpu_memory_utilization=llm_gpu_memory_utilization,
        enforce_eager=True,
        distributed_executor_backend=None,
        dtype=qcfg["dtype"],
        quantization=qcfg["quantization"],
        load_format=qcfg["load_format"],
        seed=config.seed,
    )

    gpu_mem_gb = get_gpu_memory_used()
    print(f"  GPU memory used after load: {gpu_mem_gb:.2f} GB")

    trial_times = []
    for trial_idx in range(num_trials):
        start_time = time.time()
        bon_search_v1.best_of_n_v1(batch_of_questions, config, llm_vllm, trial_idx)
        elapsed = time.time() - start_time
        trial_times.append(elapsed)
        print(f"  trial {trial_idx}: {elapsed / num_questions:.4f}s/question  {elapsed:.2f}s total")

    avg_time = sum(trial_times) / len(trial_times)
    results_summary.append((qcfg["name"], gpu_mem_gb, avg_time, avg_time / num_questions))

    del llm_vllm
    gc.collect()
    torch.cuda.empty_cache()

print("\n=== Summary ===")
print(f"{'quantization':<25} {'gpu mem (GB)':>12} {'avg s/trial':>12} {'avg s/question':>15}")
print("-" * 67)
for name, mem, avg_trial, avg_q in results_summary:
    print(f"{name:<25} {mem:>12.2f} {avg_trial:>12.2f} {avg_q:>15.4f}")